## 상황

한 음식점 사장님이 **배달의민족**과 **쿠팡이츠** 두 군데에서 장사를 하고 있습니다.
플랫폼마다 매출 파일(CSV)을 하루에 하나씩 내려받는데, 칼럼명이 서로 달라서 매번 손으로 합치고 있습니다.
식자재는 **식자재마트**에서 체크카드로 사기 때문에 재료비는 은행 거래내역에서 확인해야 합니다.

pandas로 **한 달치(2026.10.01 ~ 2026.10.31)** 데이터를 합쳐서, 사장님이 한눈에 볼 수 있는 표를 만들어 주세요.

## 문제

## 1. 파일 읽기
- `baemin` 폴더의 파일 31개를 모두 읽어서 하나의 DataFrame으로 만드세요.
- `coupang` 폴더의 파일 31개도 같은 방식으로 읽으세요.
- 파일 목록은 아래 힌트 코드로 가져오고, 반복문으로 하나씩 읽어서 합치세요.

In [36]:
import glob
import pandas as pd

In [37]:
baemin_files = sorted(glob.glob("data/baemin/baemin*.csv"))
print(len(baemin_files))

coupang_files = sorted(glob.glob("data/coupang/coupang*.csv"))
print(len(coupang_files))

baemin_list = [] # 파일 내용을 하나의 리스트로 만들기 (배 민)
for list in baemin_files:
    df = pd.read_csv(list)
    baemin_list.append(df)


baemin = pd.concat(baemin_list, ignore_index=True) # 리스트에서 DataFrame으로 묶기 (배 민)
print(baemin)


coupang_list = []               # (쿠팡)
for list in coupang_files:
    df = pd.read_csv(list)
    coupang_list.append(df)

coupang = pd.concat(coupang_list, ignore_index=True)    # (쿠팡)
print(coupang)

31
31
                 order_at  total_bill ad_yn  cpc
0     2026-10-01 11:48:34       26000     Y  700
1     2026-10-01 11:53:22       15000     N    0
2     2026-10-01 12:18:52       35000     N    0
3     2026-10-01 12:28:44       35000     N    0
4     2026-10-01 12:50:50       35000     N    0
...                   ...         ...   ...  ...
1150  2026-10-31 20:34:53       17000     N    0
1151  2026-10-31 20:47:07       38000     Y  640
1152  2026-10-31 21:39:35       19000     Y  690
1153  2026-10-31 21:45:29       45000     N    0
1154  2026-10-31 21:51:03       19000     N    0

[1155 rows x 4 columns]
              order_time  total_price advertisement_yn  ad_pct
0    2026-10-01 11:46:07        18000                Y      10
1    2026-10-01 12:06:50        21000                Y      10
2    2026-10-01 12:16:29        58000                Y      10
3    2026-10-01 12:19:49        43000                Y      10
4    2026-10-01 13:00:05        58000                N       0
.. 

### 2. 칼럼명 통일

- 두 플랫폼의 칼럼명을 `주문일시`, `전체금액`, `광고유입여부`로 바꾸세요.
- 어느 플랫폼 데이터인지 알 수 있도록 `플랫폼` 칼럼(`배민` / `쿠팡`)을 추가하세요.
- 두 플랫폼 모두 주문 1건마다 `광고집행비`(원) 칼럼을 만드세요.
    - 배민: `cpc` 그대로
    - 쿠팡: `전체금액 × ad_pct / 100`

In [38]:
# 배달의 민족 파일 정리
baemin = baemin.rename(columns={
    "order_at": "주문일시",
    "total_bill": "전체금액",
    "ad_yn": "광고유입여부",
})
baemin["플랫폼"] = "배 민"

baemin["광고집행비"] = baemin["cpc"]
baemin

,주문일시,전체금액,광고유입여부,cpc,플랫폼,광고집행비
0,2026-10-01 11:48:34,26000,Y,700,배 민,700
1,2026-10-01 11:53:22,15000,N,0,배 민,0
2,2026-10-01 12:18:52,35000,N,0,배 민,0
3,2026-10-01 12:28:44,35000,N,0,배 민,0
4,2026-10-01 12:50:50,35000,N,0,배 민,0
...,...,...,...,...,...,...
1150,2026-10-31 20:34:53,17000,N,0,배 민,0
1151,2026-10-31 20:47:07,38000,Y,640,배 민,640
1152,2026-10-31 21:39:35,19000,Y,690,배 민,690
1153,2026-10-31 21:45:29,45000,N,0,배 민,0


In [39]:
# 쿠팡 파일 정리
coupang = coupang.rename(columns={
    "order_time": "주문일시",
    "total_price": "전체금액",
    "advertisement_yn": "광고유입여부",
})
coupang["플랫폼"] = "쿠팡"

coupang["광고집행비"] = coupang["전체금액"] * coupang["ad_pct"] / 100
coupang

,주문일시,전체금액,광고유입여부,ad_pct,플랫폼,광고집행비
0,2026-10-01 11:46:07,18000,Y,10,쿠팡,1800.0
1,2026-10-01 12:06:50,21000,Y,10,쿠팡,2100.0
2,2026-10-01 12:16:29,58000,Y,10,쿠팡,5800.0
3,2026-10-01 12:19:49,43000,Y,10,쿠팡,4300.0
4,2026-10-01 13:00:05,58000,N,0,쿠팡,0.0
...,...,...,...,...,...,...
707,2026-10-31 19:33:36,32000,N,0,쿠팡,0.0
708,2026-10-31 19:36:08,18000,Y,5,쿠팡,900.0
709,2026-10-31 19:50:22,18000,N,0,쿠팡,0.0
710,2026-10-31 19:59:04,34000,N,0,쿠팡,0.0


### 3. 데이터 통합

- 두 DataFrame을 하나로 합치세요. (`pd.concat`)
- `주문일시`를 datetime으로 바꾸고, 날짜만 담은 `주문일자` 칼럼을 추가하세요.
- 주문일시 순으로 정렬하세요.

In [40]:
# 배민 , 쿠팡 병합
columns = ["플랫폼", "주문일시", "전체금액", "광고유입여부", "광고집행비"] # 공통의 칼럼 5개로 병합
orders = pd.concat([baemin[columns], coupang[columns]], ignore_index=True)

# 칼럼 변경 후 추가
orders["주문일시"] = pd.to_datetime(orders["주문일시"])
orders["주문일자"] = orders["주문일시"].dt.date

# 정렬
orders = orders.sort_values("주문일시")
orders = orders.reset_index( drop = True )

orders

,플랫폼,주문일시,전체금액,광고유입여부,광고집행비,주문일자
0,쿠팡,2026-10-01 11:46:07,18000,Y,1800.0,2026-10-01
1,배 민,2026-10-01 11:48:34,26000,Y,700.0,2026-10-01
2,배 민,2026-10-01 11:53:22,15000,N,0.0,2026-10-01
3,쿠팡,2026-10-01 12:06:50,21000,Y,2100.0,2026-10-01
4,쿠팡,2026-10-01 12:16:29,58000,Y,5800.0,2026-10-01
...,...,...,...,...,...,...
1862,배 민,2026-10-31 20:47:07,38000,Y,640.0,2026-10-31
1863,쿠팡,2026-10-31 20:49:23,24000,N,0.0,2026-10-31
1864,배 민,2026-10-31 21:39:35,19000,Y,690.0,2026-10-31
1865,배 민,2026-10-31 21:45:29,45000,N,0.0,2026-10-31


### 4. 일자별 매출 집계

- 행은 `주문일자`, 열은 `플랫폼`, 값은 `전체금액` 합계인 표를 만드세요. (`pivot_table`)
- `합계` 열을 추가하세요.

In [41]:
days_sales = orders.pivot_table(
    index="주문일자",
        columns="플랫폼",
        values="전체금액",
        aggfunc="sum",
)

days_sales["합계"] = days_sales["배 민"] + days_sales["쿠팡"]

days_sales

플랫폼,배 민,쿠팡,합계
주문일자,,,
2026-10-01,959000,562000,1521000
2026-10-02,1348000,904000,2252000
2026-10-03,1294000,779000,2073000
2026-10-04,1127000,928000,2055000
2026-10-05,1103000,543000,1646000
2026-10-06,1028000,576000,1604000
2026-10-07,1227000,549000,1776000
2026-10-08,946000,439000,1385000
2026-10-09,1200000,835000,2035000


### 5. 광고 효과 분석

- 플랫폼별로 광고유입 주문이 전체 주문 건수에서 차지하는 **비율(%)**을 구하세요.
- 플랫폼별 **광고집행비** 합계를 구하세요.

In [42]:
ad_orders = orders[orders["광고유입여부"] == "Y"]

total_count = orders.groupby("플랫폼").size()
ad_count = ad_orders.groupby("플랫폼").size()

ad_rate = ad_count / total_count * 100
print(ad_rate.round(2))

ad_money = orders.groupby("플랫폼")["광고집행비"].sum()
print(ad_money)

플랫폼
배 민    34.98
쿠팡     51.97
dtype: float64
플랫폼
배 민    243170.0
쿠팡     886700.0
Name: 광고집행비, dtype: float64


### 6. 식자재비 추출

- 은행 거래내역에서 거래처가 정확히 `식자재마트`인 출금 내역만 골라내세요.
- 일자별 식자재비 합계를 구하세요.

In [43]:
bank = pd.read_csv("data/bank_20261001_20261031_utf8.csv", encoding="utf8")

bank["거래일시"] = pd.to_datetime(bank["거래일시"], format="%Y.%m.%d %H:%M")

foods = bank[bank["거래처"] == "식자재마트"].copy() # 혹시 원본을 변경하지 않기 위해
foods["주문일자"] = foods["거래일시"].dt.date
print(foods)

days_food = foods.groupby("주문일자")["출금액"].sum()
days_food.name = "식자재비"
print(days_food)


                   거래일시    적요    거래처     출금액  입금액        잔액        주문일자
1   2026-10-01 09:01:00  체크카드  식자재마트  197100    0   4796800  2026-10-01
5   2026-10-02 09:15:00  체크카드  식자재마트  267100    0   6193900  2026-10-02
7   2026-10-02 15:03:00  체크카드  식자재마트  113300    0   6056000  2026-10-02
10  2026-10-03 09:07:00  체크카드  식자재마트  194800    0   5437000  2026-10-03
11  2026-10-03 15:28:00  체크카드  식자재마트   52800    0   5384200  2026-10-03
13  2026-10-04 09:08:00  체크카드  식자재마트  289300    0   5082700  2026-10-04
15  2026-10-05 09:44:00  체크카드  식자재마트  214400    0   4859900  2026-10-05
20  2026-10-06 09:04:00  체크카드  식자재마트  209500    0   6969700  2026-10-06
24  2026-10-07 09:03:00  체크카드  식자재마트  273100    0   6540300  2026-10-07
26  2026-10-07 15:16:00  체크카드  식자재마트   59700    0   6470700  2026-10-07
28  2026-10-08 09:02:00  체크카드  식자재마트  219300    0   6245800  2026-10-08
32  2026-10-09 09:28:00  체크카드  식자재마트  257300    0   7656800  2026-10-09
34  2026-10-09 15:46:00  체크카드  식자재마트   94000    0   7542400  202

### 7. 최종 결과표 저장

- 4번 결과와 6번 결과를 일자 기준으로 합쳐서 아래 칼럼을 가진 표를 만드세요.
   / 주문일자	/ 배민	/ 쿠팡	/ 합계	/ 식자재비	/ 매출-식자재비
- 결과를 result.csv로 저장하세요. (엑셀에서 열어도 한글이 안 깨지게 encoding="utf-8-sig")

In [44]:
result = days_sales.join(days_food)
result["매출-식자재비"] = result["합계"] - result["식자재비"]

result = result.reset_index()
result

result.to_csv("result.csv", index=False, encoding="utf-8-sig")